In [26]:
import pandas as pd
import torch
import torch.nn as nn
import khmernltk
import pickle
import re
from tqdm import tqdm

# --- 1. Define the Cleaning Function ---
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = text.replace('_', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# --- 2. Load the Vocabulary ---
with open('vocab2.pkl', 'rb') as f:
    vocab = pickle.load(f)
word2idx = vocab['word2idx']
char2idx = vocab['char2idx']
tag2idx = vocab['tag2idx']
idx2tag = {v: k for k, v in tag2idx.items()}

# --- 3. Define Model Architecture ---
class CharCNN_BiLSTM_Labeler(nn.Module):
    def __init__(self, word_vocab_size, char_vocab_size, tagset_size, 
                 word_embed_dim=128, char_embed_dim=50, char_filters=50, 
                 kernel_size=3, hidden_dim=256):
        super(CharCNN_BiLSTM_Labeler, self).__init__()
        self.word_embedding = nn.Embedding(word_vocab_size, word_embed_dim, padding_idx=0)
        self.char_embedding = nn.Embedding(char_vocab_size, char_embed_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(char_embed_dim, char_filters, kernel_size, padding=kernel_size//2)
        lstm_input_dim = word_embed_dim + char_filters
        self.lstm = nn.LSTM(lstm_input_dim, hidden_dim // 2, num_layers=1, bidirectional=True, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, words, chars):
        batch_size, seq_len, max_word_len = chars.shape
        chars_flat = chars.view(-1, max_word_len)
        char_embeds = self.char_embedding(chars_flat).transpose(1, 2)
        cnn_out = torch.relu(self.char_cnn(char_embeds))
        char_features, _ = torch.max(cnn_out, dim=2)
        char_features = char_features.view(batch_size, seq_len, -1)
        word_features = self.word_embedding(words)
        combined_features = torch.cat((word_features, char_features), dim=2)
        lstm_out, _ = self.lstm(combined_features)
        return self.hidden2tag(lstm_out)

# --- 4. Load the Weights ---
cs_model = CharCNN_BiLSTM_Labeler(len(word2idx), len(char2idx), len(tag2idx))
cs_model.load_state_dict(torch.load('khmer_cs_char_cnn_model.pth', map_location=torch.device('cpu')))
cs_model.eval()

# --- 5. Define the Prediction Helper ---
def predict_language(comment, word2idx, char2idx, model):
    tokens = comment.strip().split()
    if not tokens: return "OTHER"
    
    word_ids = [word2idx.get(w, word2idx.get('<UNK>', 1)) for w in tokens]
    max_word_len = max(len(w) for w in tokens)
    char_ids = [[char2idx.get(c, char2idx.get('<UNK>', 1)) for c in w] + [0]*(max_word_len-len(w)) for w in tokens]
    
    words_tensor = torch.tensor([word_ids], dtype=torch.long)
    chars_tensor = torch.tensor([char_ids], dtype=torch.long)
    
    with torch.no_grad():
        outputs = model(words_tensor, chars_tensor)
        prediction_indices = torch.argmax(outputs, dim=2)[0].tolist()
        
    most_common_tag = max(set(prediction_indices), key=prediction_indices.count)
    return idx2tag[most_common_tag]

# --- 6. Load Dataset ---
df = pd.read_csv('all_data.csv')  # Columns: 'Comment', 'Label', 'label_id', 'source'

processed_texts = []
lang_tags = []

print("Processing dataset...")
for comment in tqdm(df['Comment']):

    # A. Clean the text FIRST
    cleaned_comment = clean_text(comment)

    if not cleaned_comment:
        processed_texts.append("")
        lang_tags.append("OTHER")
        continue

    # B. ✅ Actually use the cs_model to predict language tag
    predicted_tag = predict_language(cleaned_comment, word2idx, char2idx, cs_model)
    lang_tags.append(predicted_tag)

    # C. Segment words using KhmerNLTK
    tokens = khmernltk.word_tokenize(cleaned_comment)
    spaced_comment = " ".join(tokens)

    # D. Clean up double spaces
    final_spaced_comment = re.sub(r'\s+', ' ', spaced_comment).strip()

    # E. Fuse the Tag and the Segmented Text
    fused_text = f"[{predicted_tag}] {final_spaced_comment}"
    processed_texts.append(fused_text)

# Save preprocessed dataset
df['Processed_Comment'] = processed_texts
df['Lang_Tag'] = lang_tags
df.to_csv('segmented_tagged_dataset_cleaned.csv', index=False)
print("Data preprocessing complete! File saved.")

C:\Users\USER\AppData\Local\Temp\ipykernel_10956\859655144.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cs_model.load_state_dict(torch.load('khmer_cs_char_cnn_model.

Processing dataset...


100%|██████████| 7019/7019 [00:23<00:00, 302.89it/s]

Data preprocessing complete! File saved.


In [27]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Load preprocessed data
df = pd.read_csv('segmented_tagged_dataset_cleaned.csv')

# ✅ Fixed: correct column names (lowercase)
df = df.dropna(subset=['Processed_Comment', 'label_id'])

texts = df['Processed_Comment'].astype(str).tolist()
labels = df['label_id'].astype(int).tolist()  # ✅ Fixed: use label_id (0=Neg, 1=Neu, 2=Pos)

# Train/Val Split (80/20)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# Initialize XLM-RoBERTa Tokenizer
print("Loading XLM-RoBERTa Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# Tokenize
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")

# Create PyTorch Datasets
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, train_labels)
val_dataset = SentimentDataset(val_encodings, val_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

Loading XLM-RoBERTa Tokenizer...


In [28]:
import torch.nn as nn
from sklearn.metrics import f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load the base model (num_labels=3 for Positive, Negative, Neutral)
print("Loading XLM-RoBERTa Model...")
model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3)
model.to(device)

# Handle Class Imbalance
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

# Optimizer (Use a very small learning rate for Transformers!)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# Training Hyperparameters
epochs = 5
patience = 2
best_f1 = 0
epochs_no_improve = 0

print("Starting training...")
for epoch in range(epochs):
    
    # --- Training Phase ---
    model.train()
    total_train_loss = 0
    
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --- Validation Phase ---
    model.eval()
    total_val_loss = 0
    all_preds, all_true = [], []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            total_val_loss += loss.item()
            
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(labels.cpu().numpy())
            
    avg_val_loss = total_val_loss / len(val_loader)
    macro_f1 = f1_score(all_true, all_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {macro_f1:.4f}")
    
    # --- Early Stopping (Based on F1 Score) ---
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_xlmr_sentiment_model.pth')
        print("  ✓ Val F1-Score improved → model saved.")
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{patience})")
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}.")
            break

print(f"\nTraining Complete. Best Validation Macro F1: {best_f1:.4f}")

Loading XLM-RoBERTa Model...


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting training...


KeyboardInterrupt: 